# `cind` — profile, choose columns & trim

Profile every column, then choose **8 columns** (cardinality mix + id/super-key, drop constants), trim to **45,000 rows** (Table 14). Save `cind_8c_45000r.csv` in this folder.

> **Note:** output columns are always `c0`…`c7`; selection keeps a cardinality mix and one id/super-key column when present.

In [1]:
import os
import numpy as np
import pandas as pd

NAME       = "cind"
N_ROWS     = 45000      # exact target rows (Table 14)
N_COLS     = 8         # exact target cols (Table 14)

RAW_PATH   = "CIND.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ";"
HAS_HEADER = False
ENCODING   = "utf-8"
OUT_DELIM   = ","                 # trimmed CSV always comma-separated
ON_BAD_LINES = "skip"              # CIND.csv has ragged rows


## 1. View the raw data

In [2]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (2247445, 10)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9
0,NaN,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete,NaN
1,NaN,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete,NaN
2,NaN,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete,NaN
3,NaN,ENT_OID=0000000000000000000000002355930.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete,NaN
4,NaN,ENT_OID=0000000000000000000000002356058.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete,NaN


In [3]:
raw.dtypes

c0    float64
c1     object
c2     object
c3     object
c4     object
c5     object
c6     object
c7     object
c8     object
c9    float64
dtype: object

## 2. Profile: cardinality, top-value %, and group skew

In [4]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col,
        distinct,
        nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2),
        int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,0,2247445,100.0,0.00,100.00,2247445.00,2247445,1.00
c1,2208706,0,0.0,98.28,0.00,1.02,5,4.91
c2,2,0,0.0,0.00,100.00,1123722.50,2247440,2.00
c3,5,0,0.0,0.00,51.32,449489.00,1153383,2.57
c4,5,0,0.0,0.00,51.32,449489.00,1153379,2.57
c5,2,0,0.0,0.00,100.00,1123722.50,2247440,2.00
c6,3,0,0.0,0.00,91.48,749148.33,2055962,2.74
c7,2,0,0.0,0.00,100.00,1123722.50,2247440,2.00
c8,3,1,0.0,0.00,96.55,561861.25,2169982,3.86


## 3. Choose columns (cardinality mix + id/super-key)

In [5]:
prof_sel = prof[prof.distinct_values > 1]

KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]

chosen = []

if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])

n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0:
                break

for c in raw.columns:
    if len(chosen) >= N_COLS:
        break
    if c not in chosen:
        chosen.append(c)

SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8']


## 4. Trim to the chosen columns x exact rows

First `N_ROWS` rows of `SELECTED_COLS`. For a distribution-preserving sample instead of the head, use the commented `.sample(...)` line.

In [6]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
assert raw.shape[0] >= N_ROWS, f"need >= {N_ROWS} rows, have {raw.shape[0]}"

trimmed = raw.loc[:, SELECTED_COLS].iloc[:N_ROWS].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]
# trimmed = raw.loc[:, SELECTED_COLS].sample(N_ROWS, random_state=42).reset_index(drop=True)
# trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (N_ROWS, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (45000, 8)


,c0,c1,c2,c3,c4,c5,c6,c7
0,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
1,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
2,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
3,ENT_OID=0000000000000000000000002355930.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete
4,ENT_OID=0000000000000000000000002356058.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete


## 5. Save the trimmed CSV

In [7]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote WIKIRANK_8c_45000r.csv (45000, 8)
reloaded: (45000, 8)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7']


## 6. Check selected cardinality and skew

In [8]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

output shape : (45000, 8)
columns      : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7']


,c0,c1,c2,c3,c4,c5,c6,c7
0,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
1,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
2,A=1,WIKIRANK,R,B,WIKIRANK,S,C,totally complete
3,ENT_OID=0000000000000000000000002355930.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete
4,ENT_OID=0000000000000000000000002356058.,BIOSQLSP,SG_BIOENTRY_QUALIFIER_ASSOC,TRM_OID,BIOSQLSP,SG_TERM,OID,valid & complete


In [9]:
total_rows = len(output)

rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col,
        distinct,
        nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2),
        int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

out_prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,38553,0,0.0,85.67,0.01,1.17,5,4.28
c1,2,0,0.0,0.00,99.99,22500.00,44995,2.00
c2,3,0,0.0,0.01,99.54,15000.00,44793,2.99
c3,3,0,0.0,0.01,99.54,15000.00,44793,2.99
c4,2,0,0.0,0.00,99.99,22500.00,44995,2.00
c5,2,0,0.0,0.00,99.99,22500.00,44995,2.00
c6,2,0,0.0,0.00,99.99,22500.00,44995,2.00
c7,3,1,0.0,0.01,99.54,11250.00,44794,3.98
